In [1]:
import sys
sys.path.append('..')

import pandas as pd
from src.limpeza import carregar_tabelas, preencher_categoria, remover_duplicadas

tabelas = carregar_tabelas()
print(tabelas.keys())

dict_keys(['customers', 'geolocation', 'order_items', 'order_payments', 'order_reviews', 'orders', 'products', 'sellers', 'category_translation'])


In [2]:
for nome, df in tabelas.items():
    nulos = df.isna().sum()
    nulos = nulos[nulos > 0]
    if len(nulos) > 0:
        print(nome)
        print(nulos)
        print()

order_reviews
review_comment_title      87656
review_comment_message    58247
dtype: int64

orders
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
dtype: int64

products
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64



In [3]:
pedidos = tabelas["orders"]

entregues = pedidos[pedidos["order_status"] == "delivered"]

entregues[["order_approved_at", "order_delivered_carrier_date", "order_delivered_customer_date"]].isna().sum()

order_approved_at                14
order_delivered_carrier_date      2
order_delivered_customer_date     8
dtype: int64

In [4]:
for nome, df in tabelas.items():
    print(nome, df.duplicated().sum())

customers 0
geolocation 261831
order_items 0
order_payments 0
order_reviews 0
orders 0
products 0
sellers 0
category_translation 0


In [5]:
pedidos = tabelas["orders"].copy()

pedidos["compra"] = pd.to_datetime(pedidos["order_purchase_timestamp"])
pedidos["entrega"] = pd.to_datetime(pedidos["order_delivered_customer_date"])

impossiveis = pedidos[pedidos["entrega"] < pedidos["compra"]]
print(len(impossiveis))

0


In [6]:
pedidos["aprovacao"] = pd.to_datetime(pedidos["order_approved_at"])
pedidos["transportadora"] = pd.to_datetime(pedidos["order_delivered_carrier_date"])

antes_compra = pedidos[pedidos["transportadora"] < pedidos["compra"]]
antes_aprovar = pedidos[pedidos["transportadora"] < pedidos["aprovacao"]]

print("Transportadora antes da compra:", len(antes_compra))
print("Transportadora antes da aprovação:", len(antes_aprovar))

Transportadora antes da compra: 166
Transportadora antes da aprovação: 1359


In [7]:
produtos = preencher_categoria(tabelas["products"])
print(produtos["product_category_name"].isna().sum())
print((produtos["product_category_name"] == "sem_categoria").sum())

geo = remover_duplicadas(tabelas["geolocation"])
print(len(tabelas["geolocation"]), "→", len(geo))

0
610
1000163 → 738332
